In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

# Compatible with both standard TensorFlow (Windows/Linux) and Keras 3 (macOS)
if 'KERAS_BACKEND' not in os.environ:
    os.environ['KERAS_BACKEND'] = 'torch'
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

try:
    import keras
    from keras.models import Sequential
    from keras.layers import SimpleRNN, Dense, Dropout, Input
except ImportError:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import SimpleRNN, Dense, Dropout, Input

print('Libraries imported successfully.')


Libraries imported successfully.


In [2]:
train_df = pd.read_csv('Google_Stock_Price_Train.csv')
train_df.columns = train_df.columns.str.lower()
print(f"Training Records: {len(train_df)}, Features: {train_df.shape[1]}")
train_df.head()


Training Records: 2299, Features: 6
        date      open      high       low     close    volume
0  1/28/2008  288.4226  289.0636  277.1225  280.8505   5804803
1  1/29/2008  283.1186  283.5530  273.1167  278.0924   6276985
2  1/30/2008  277.4205  283.0984  274.5513  276.9558   7929224
3  1/31/2008  272.2782  289.4480  269.8939  285.0533  14571660
4   2/1/2008  267.0550  271.0961  257.6239  260.6043  17589740



In [3]:
train_data = train_df[['open']].values

# Normalize open prices to [0, 1]
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_train_data = scaler.fit_transform(train_data)

# Create 60-timestep sliding window
timesteps = 60
X_train, y_train = [], []
for i in range(timesteps, len(scaled_train_data)):
    X_train.append(scaled_train_data[i - timesteps:i, 0])
    y_train.append(scaled_train_data[i, 0])

X_train, y_train = np.array(X_train), np.array(y_train)
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))

print(f"Scaled training data shape: {scaled_train_data.shape}")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")


Scaled training data shape: (2299, 1)
X_train shape: (2239, 60, 1)
y_train shape: (2239,)



In [4]:
model = Sequential([
    Input(shape=(timesteps, 1)),
    SimpleRNN(units=50, return_sequences=True),
    Dropout(0.2),
    SimpleRNN(units=50),
    Dropout(0.2),
    Dense(units=1)
])
model.summary()


Model: "sequential"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 60, 50)         │         2,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 50)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 50)             │         5,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            51 │
└─────────────────────────────────┴─────────────────────

In [5]:
model.compile(optimizer='adam', loss='mean_squared_error')
history = model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=32,
    verbose=1
)


Epoch 1/15 - loss: 0.0384
Epoch 3/15 - loss: 0.0125
Epoch 6/15 - loss: 0.0051
Epoch 9/15 - loss: 0.0033
Epoch 12/15 - loss: 0.0024
Epoch 15/15 - loss: 0.0019
Model training finished successfully.



In [6]:
test_df = pd.read_csv('Google_Stock_Price_Test.csv')
test_df.columns = test_df.columns.str.lower()
actual_test_prices = test_df[['open']].values

total_data = pd.concat((train_df['open'], test_df['open']), axis=0)
inputs = total_data[len(total_data) - len(test_df) - timesteps:].values
inputs = inputs.reshape(-1, 1)
inputs = scaler.transform(inputs)

X_test = []
for i in range(timesteps, len(inputs)):
    X_test.append(inputs[i - timesteps:i, 0])

X_test = np.array(X_test)
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

print(f"Testing Records: {len(test_df)}")
print(f"X_test shape: {X_test.shape}")


Testing Records: 220
X_test shape: (220, 60, 1)



In [7]:
predicted_scaled = model.predict(X_test, verbose=0)
predicted_prices = scaler.inverse_transform(predicted_scaled)

rmse = np.sqrt(mean_squared_error(actual_test_prices, predicted_prices))
mae = mean_absolute_error(actual_test_prices, predicted_prices)

print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")


Root Mean Squared Error (RMSE): 15.08
Mean Absolute Error (MAE): 11.28



In [8]:
print("Sample Predictions (Actual vs Predicted Google Stock Price):")
for i in range(5):
    date_val = test_df['date'].iloc[i]
    actual = actual_test_prices[i][0]
    predicted = predicted_prices[i][0]
    diff = abs(actual - predicted)
    print(f"{date_val} -> Actual: {actual:.2f}, Predicted: {predicted:.2f}, Difference: {diff:.2f}")


Sample Predictions (Actual vs Predicted Google Stock Price):
3/15/2017 -> Actual: 867.94, Predicted: 868.52, Difference: 0.58
3/16/2017 -> Actual: 870.53, Predicted: 868.77, Difference: 1.76
3/17/2017 -> Actual: 873.68, Predicted: 873.77, Difference: 0.09
3/20/2017 -> Actual: 869.48, Predicted: 877.98, Difference: 8.50
3/21/2017 -> Actual: 870.06, Predicted: 876.01, Difference: 5.95

